In [ ]:
!nvidia-smi

Sun Sep  1 09:55:10 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P0              44W / 400W |      2MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.3/872.3 kB 46.7 MB/s eta 0:00:00


In [ ]:
import os
from ultralytics import YOLO

# Directory for your WLI data
DATA_DIR_wli = '/content/drive/MyDrive/sk_split_data/save'

# List of YOLOv8 model versions you want to try
model_versions = ["yolov8x-cls.pt", "yolov8n-cls.pt"]  # YOLOv8 'x' and 'n' versions

# Initialize variables to track the best model
best_model = None
best_accuracy = 0

# Train each model version
for model_version in model_versions:
    model = YOLO(model_version)  # Load the pretrained model

    # Train in increments of 75 epochs, up to 150 epochs
    for i in range(0, 150, 75):
        # Specify a unique directory for each 100-epoch segment
        run_name = f"train_{model_version.split('.')[0]}_{i+100}_epochs"
        results = model.train(
            data=DATA_DIR_wli,
            epochs=100,           # Train for 100 epochs
            imgsz=640,            # Image size
            batch=16,             # Batch size
            augment=True,         # Enable augmentation
            flipud=0.5,           # Vertical flip with 50% probability
            fliplr=0.5,           # Horizontal flip with 50% probability
            degrees=10,           # Rotate images by up to 10 degrees
            scale=0.5,            # Scale images by up to 50%
            translate=0.1,        # Translate images by 10% in x and y directions
            lr0=0.001,            # Initial learning rate
            save=True,            # Save the model checkpoints
            project="runs",       # Save directory
            name=run_name         # Unique name for the run
        )

        # Evaluate the model on the validation set
        val_results = model.val(data=DATA_DIR_wli, imgsz=640, batch=16)

        # Track accuracy or any other metric
        current_accuracy = val_results.top1  # Access accuracy directly from val_results
        # You can also access top5 accuracy using val_results.top5

        if current_accuracy > best_accuracy:
            best_accuracy = current_accuracy
            best_model = model_version

        # Log the results
        print(f"Model: {model_version}, Epochs: {i + 75}, Accuracy: {current_accuracy}")

# Output the best model after all training
print(f"Best model: {best_model} with accuracy: {best_accuracy}")

100%|██████████| 110M/110M [00:00<00:00, 308MB/s] 


Ultralytics YOLOv8.2.85 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=classify, mode=train, model=yolov8x-cls.pt, data=/content/drive/MyDrive/sk_split_data/save, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=runs, name=train_yolov8x-cls_100_epochs, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False,

100%|██████████| 6.25M/6.25M [00:00<00:00, 353MB/s]


AMP: checks passed ✅


/usr/local/lib/python3.10/dist-packages/ultralytics/engine/trainer.py:268: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning /content/drive/MyDrive/sk_split_data/save/train... 796 images, 0 corrupt: 100%|██████████| 796/796 [01:35<00:00,  8.33it/s]


train: New cache created: /content/drive/MyDrive/sk_split_data/save/train.cache


... (347 outputs left out)


      Epoch    GPU_mem       loss  Instances       Size


      8/100      9.83G      1.004         12        640: 100%|██████████| 50/50 [00:12<00:00,  4.01it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 5/5 [00:00<00:00, 12.21it/s]

                   all      0.323          1


KeyboardInterrupt: 

In [ ]:


from sklearn.metrics import classification_report


confusion_matrix = [
    [48, 9, 3],
    [1, 40, 4],
    [3, 4, 48]
]

# Generate classification report
report = classification_report(
    [0] * sum(confusion_matrix[0]) + [1] * sum(confusion_matrix[1]) + [2] * sum(confusion_matrix[2]),
    [0 for _ in range(confusion_matrix[0][0])] + [1 for _ in range(confusion_matrix[0][1])] + [2 for _ in range(confusion_matrix[0][2])] +
    [0 for _ in range(confusion_matrix[1][0])] + [1 for _ in range(confusion_matrix[1][1])] + [2 for _ in range(confusion_matrix[1][2])] +
    [0 for _ in range(confusion_matrix[2][0])] + [1 for _ in range(confusion_matrix[2][1])] + [2 for _ in range(confusion_matrix[2][2])]
)

print(report)


              precision    recall  f1-score   support

           0       0.92      0.80      0.86        60
           1       0.75      0.89      0.82        45
           2       0.87      0.87      0.87        55

    accuracy                           0.85       160
   macro avg       0.85      0.85      0.85       160
weighted avg       0.86      0.85      0.85       160



In [ ]:
# prompt: download this folder /content/runs/train_yolov8x-cls_100_epochs/weights

!zip -r /content/runs_train_yolov8x-cls_100_epochs_weights.zip /content/runs/train_yolov8x-cls_100_epochs/weights
from google.colab import files
files.download("/content/runs_train_yolov8x-cls_100_epochs_weights.zip")


  adding: content/runs/train_yolov8x-cls_100_epochs/weights/ (stored 0%)
  adding: content/runs/train_yolov8x-cls_100_epochs/weights/best.pt (deflated 8%)
  adding: content/runs/train_yolov8x-cls_100_epochs/weights/last.pt (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>